In [ ]:
import pandas as pd
import os
import psycopg2
import warnings
warnings.filterwarnings('ignore')

trend_path = r"C:\Users\Administrator\Desktop\data_learn\Hot-Drink-Retail-Analysis\data\processed\simulated_trend_1year.csv"
weather_path = r"C:\Users\Administrator\Desktop\data_learn\Hot-Drink-Retail-Analysis\data\raw\beijing_weather.csv"

trend_df = pd.read_csv(trend_path)
weather_df = pd.read_csv(weather_path)



In [ ]:
trend_df['date'] = pd.to_datetime(trend_df['date'])
weather_df['time'] = pd.to_datetime(weather_df['time'])

merge_df = pd.merge(
    left=trend_df,
    right=weather_df,
    left_on='date',
    right_on='time',
    how='inner'
)

result_path = r"C:\Users\Administrator\Desktop\data_learn\Hot-Drink-Retail-Analysis\data\processed\temperature_trend_merged_data.csv"
merge_df.to_csv(result_path, index=False, encoding='utf-8-sig')

print(f'趋势表的行数: {len(trend_df)}')
print(f'天气表的行数: {len(weather_df)}')
print(f'合并后表的行数: {len(merge_df)}')


In [ ]:
conn = psycopg2.connect(
    host = 'localhost',
    port = '5432',
    database = 'postgres',
    user = 'postgres',
    password = ''
)

cur = conn.cursor()

query_create ="""
DROP TABLE IF EXISTS hot_drink_retail.temperature_trend_merged_data;

CREATE TABLE hot_drink_retail.temperature_trend_merged_data AS
SELECT 
    t.*,
    w.tavg,
    w.tmin,
    w.tmax
FROM hot_drink_retail.simulated_trend_1year AS t
INNER JOIN hot_drink_retail.beijing_weather AS w
    ON t.date::DATE = w.time::DATE
"""

cur.execute(query_create)
conn.commit()

df_sql = pd.read_sql_query('SELECT * FROM hot_drink_retail.temperature_trend_merged_data LIMIT 5', conn)

cur.close()

display(df_sql.head())

